# 17 — Hierarchical N=16 Synchronisation

UPDE tuning showed: flat Kuramoto desynchronises at N≥12 with K=2.7, alpha=0.3.
The SCPN claims 16 layers. Either:
- The SCPN is broken (16 layers can't sync)
- Or hierarchical domain structure is NECESSARY for full-system sync

**Test**: Does domain-local coupling (6 domains, 2-3 layers each) plus
cross-domain bridges allow N=16 to synchronise where flat topology fails?

In [ ]:
import json

import numpy as np

from scpn_quantum_control.bridge import OMEGA_N_16, build_knm_paper27

In [ ]:
# SCPN Domain structure (Paper 0)
DOMAINS = {
    "I_biological": [0, 1, 2, 3],  # L1-L4
    "II_organismal": [4, 5, 6, 7],  # L5-L8
    "III_memory": [8, 9],  # L9-L10
    "IV_collective": [10, 11],  # L11-L12
    "V_meta": [12, 13, 14],  # L13-L15
    "VI_closure": [15],  # L16 (Director)
}

N = 16
omega = OMEGA_N_16[:N]
K_paper27 = build_knm_paper27(L=N)

print("Domain structure:")
for name, layers in DOMAINS.items():
    print(f"  {name:20s}: L{[li + 1 for li in layers]}")

In [ ]:
def simulate_kuramoto_16(K, omega, T=200.0, dt=0.02, noise=0.05, n_trials=10):
    """Vectorised Kuramoto for N=16."""
    N = len(omega)
    n_steps = int(T / dt)
    R_trials = []

    for trial in range(n_trials):
        rng = np.random.default_rng(trial)
        theta = rng.uniform(0, 2 * np.pi, N)
        R_tail = []

        for s in range(n_steps):
            diff = theta[None, :] - theta[:, None]
            coupling = np.sum(K * np.sin(diff), axis=1) / N
            theta += dt * (omega + coupling) + np.sqrt(dt) * noise * rng.normal(size=N)

            if s >= n_steps * 3 // 4:
                R_tail.append(float(np.abs(np.mean(np.exp(1j * theta)))))

        R_trials.append(np.mean(R_tail))

    return np.mean(R_trials), np.std(R_trials)


def domain_R(theta, domains):
    """Per-domain order parameters."""
    result = {}
    for name, layers in domains.items():
        z = np.mean(np.exp(1j * theta[layers]))
        result[name] = float(np.abs(z))
    return result

## Test 1: Flat K_nm at N=16 (confirm desync)

In [ ]:
print("=== TEST 1: FLAT K_nm (Paper 27) ===")
K_scales = [1.0, 2.0, 3.0, 5.0, 8.0, 12.0]
print(f"{'K_scale':>8} {'R_global':>10} {'Status'}")
print("-" * 32)
for k in K_scales:
    R, R_std = simulate_kuramoto_16(K_paper27 * k, omega)
    status = "SYNC" if R > 0.5 else "DESYNC"
    print(f"{k:8.1f} {R:10.3f} +/- {R_std:.3f}  {status}")

## Test 2: Hierarchical Coupling

Strong intra-domain coupling (K_local) + weak inter-domain bridges (K_bridge).
Each domain syncs internally first, then domains sync via bridges.

In [ ]:
def build_hierarchical_K(N, domains, K_local, K_bridge, alpha_local=0.1, alpha_bridge=0.3):
    """Hierarchical coupling: strong within domain, weak between."""
    K = np.zeros((N, N))

    # Intra-domain: strong, low alpha (nearly uniform within domain)
    for _name, layers in domains.items():
        for i in layers:
            for j in layers:
                if i != j:
                    d = abs(i - j)
                    K[i, j] = K_local * np.exp(-alpha_local * d)

    # Inter-domain bridges: weak, connecting boundary layers
    domain_list = list(domains.values())
    for d_idx in range(len(domain_list) - 1):
        # Bridge: last layer of domain d connects to first layer of domain d+1
        i = domain_list[d_idx][-1]
        j = domain_list[d_idx + 1][0]
        K[i, j] = K_bridge
        K[j, i] = K_bridge

    # L16 (Director) connects back to L1 (cybernetic closure)
    K[15, 0] = K_bridge * 0.5
    K[0, 15] = K_bridge * 0.5

    return K


print("=== TEST 2: HIERARCHICAL COUPLING ===")
print(f"{'K_local':>8} {'K_bridge':>9} {'R_global':>10} {'Status'}")
print("-" * 42)

for K_local in [2.0, 5.0, 10.0, 20.0]:
    for K_bridge in [0.1, 0.5, 1.0, 2.0]:
        K_hier = build_hierarchical_K(N, DOMAINS, K_local, K_bridge)
        R, R_std = simulate_kuramoto_16(K_hier, omega)
        status = "SYNC" if R > 0.5 else "DESYNC"
        marker = " ***" if R > 0.5 else ""
        print(f"{K_local:8.1f} {K_bridge:9.1f} {R:10.3f} +/- {R_std:.3f}  {status}{marker}")

## Test 3: Domain-Local Sync (do domains sync internally?)

In [ ]:
print("=== TEST 3: PER-DOMAIN SYNC (K_local=10, K_bridge=1) ===")
K_hier = build_hierarchical_K(N, DOMAINS, K_local=10.0, K_bridge=1.0)

# Run one trial and track per-domain R
rng = np.random.default_rng(0)
theta = rng.uniform(0, 2 * np.pi, N)
dt = 0.02
n_steps = 10000

# Snapshot at intervals
snapshots = [100, 500, 1000, 2000, 5000, 10000]
snap_idx = 0

print(f"{'Step':>6}", end="")
for name in DOMAINS:
    print(f"  {name[:8]:>8}", end="")
print(f"  {'GLOBAL':>8}")
print("-" * 72)

for s in range(n_steps):
    diff = theta[None, :] - theta[:, None]
    coupling = np.sum(K_hier * np.sin(diff), axis=1) / N
    theta += dt * (omega + coupling) + np.sqrt(dt) * 0.05 * rng.normal(size=N)

    if snap_idx < len(snapshots) and s + 1 == snapshots[snap_idx]:
        d_R = domain_R(theta, DOMAINS)
        R_global = float(np.abs(np.mean(np.exp(1j * theta))))
        print(f"{s + 1:6d}", end="")
        for name in DOMAINS:
            print(f"  {d_R[name]:8.3f}", end="")
        print(f"  {R_global:8.3f}")
        snap_idx += 1

## Test 4: Flat vs Hierarchical at Same Total Coupling

In [ ]:
# Fair comparison: same total coupling energy
print("=== TEST 4: FLAT vs HIERARCHICAL (same total K) ===")
print()

for K_total_target in [50, 100, 200, 500]:
    # Flat: scale K_paper27 to match total
    K_flat_sum = np.sum(K_paper27)
    K_flat = K_paper27 * (K_total_target / K_flat_sum)

    # Hierarchical: distribute total between local and bridge
    # 80% local, 20% bridge
    K_hier_trial = build_hierarchical_K(N, DOMAINS, K_local=10.0, K_bridge=1.0)
    K_hier_sum = np.sum(K_hier_trial)
    K_hier = K_hier_trial * (K_total_target / max(K_hier_sum, 1e-10))

    R_flat, _ = simulate_kuramoto_16(K_flat, omega, n_trials=5)
    R_hier, _ = simulate_kuramoto_16(K_hier, omega, n_trials=5)

    winner = "HIER" if R_hier > R_flat + 0.05 else ("FLAT" if R_flat > R_hier + 0.05 else "TIE")
    print(f"  Total K={K_total_target:4d}: R_flat={R_flat:.3f}, R_hier={R_hier:.3f}  -> {winner}")

print()
print("If HIER wins at lower total K: hierarchy is more EFFICIENT at synchronising.")
print("This would mean the SCPN domain structure is not arbitrary —")
print("it solves the N=16 sync problem that flat coupling cannot.")

In [ ]:
print("--- JSON RESULTS ---")
# Collect summary
K_hier_best = build_hierarchical_K(N, DOMAINS, K_local=10.0, K_bridge=1.0)
R_flat_ref, _ = simulate_kuramoto_16(K_paper27 * 5.0, omega, n_trials=5)
R_hier_ref, _ = simulate_kuramoto_16(K_hier_best, omega, n_trials=5)

print(
    json.dumps(
        {
            "n_layers": N,
            "flat_K_scale_5_R": round(R_flat_ref, 4),
            "hierarchical_R": round(R_hier_ref, 4),
            "hierarchy_advantage": round(R_hier_ref - R_flat_ref, 4),
            "domains": {k: [li + 1 for li in v] for k, v in DOMAINS.items()},
        },
        indent=2,
    )
)